# LeetCode 2282 — Number of People That Can Be Seen in a Grid bằng Python

Link đề: https://leetcode.com/problems/number-of-people-that-can-be-seen-in-a-grid/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Matrix, Monotonic Stack

> **Bài Premium:** đề bài trong notebook được viết theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc và chữ ký hàm.

Skeleton LeetCode:

```python
class Solution:
    def seePeople(self, heights: List[List[int]]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

1. Tách bài 2 chiều thành **các bài 1 chiều độc lập**: mỗi người chỉ nhìn **sang phải** (cùng hàng) và **xuống dưới** (cùng cột) ⇒ đáp án = kết quả trên hàng + kết quả trên cột.
2. Giải bài 1 chiều "mỗi người nhìn thấy bao nhiêu người bên phải" (giống LC 1944) bằng **ngăn xếp đơn điệu** duyệt từ phải sang trái.
3. Xử lý tinh tế **chiều cao bằng nhau**: nhìn thấy người cao bằng mình đầu tiên nhưng không nhìn xuyên qua họ.
4. So sánh với vét cạn O(n²) mỗi dòng (theo dõi "người cao nhất ở giữa") và chứng minh độ phức tạp **khấu hao** O(n) của ngăn xếp.
5. Kiểm chứng bằng bộ vét cạn theo đúng định nghĩa trên lưới nhỏ ngẫu nhiên.

## 2. Đề bài & ví dụ

Lưới `heights` `m × n` gồm các số nguyên dương; `heights[i][j]` là chiều cao người đứng ở `(i, j)`.

Người ở `(r1, c1)` **nhìn thấy** người ở `(r2, c2)` nếu:
- `(r2, c2)` ở **bên phải** cùng hàng (`r1 == r2`, `c1 < c2`) **hoặc** ở **bên dưới** cùng cột (`r1 < r2`, `c1 == c2`);
- **mọi người ở giữa** đều **thấp hơn cả hai** người.

Trả về ma trận `answer` với `answer[i][j]` = số người mà người `(i, j)` nhìn thấy.

**Ví dụ 1:** `heights = [[3,1,4,2,5]]` → `[[2,1,2,1,0]]`. Người cao 3 thấy 1 và 4 (không thấy 5 vì 4 ở giữa không thấp hơn 3).

**Ví dụ 2:** `heights = [[5,1],[3,1],[4,1]]` → `[[3,1],[2,1],[1,0]]`.

**Ràng buộc:** `1 ≤ m, n ≤ 400`, `1 ≤ heights[i][j] ≤ 10⁵`.

In [1]:
from typing import List
import random, time

examples = [([[3, 1, 4, 2, 5]], [[2, 1, 2, 1, 0]]), ([[5, 1], [3, 1], [4, 1]], [[3, 1], [2, 1], [1, 0]])]
for h, exp in examples:
    print("heights:"); [print("   ", r) for r in h]
    print("kỳ vọng:"); [print("   ", r) for r in exp]

heights:
    [3, 1, 4, 2, 5]
kỳ vọng:
    [2, 1, 2, 1, 0]
heights:
    [5, 1]
    [3, 1]
    [4, 1]
kỳ vọng:
    [3, 1]
    [2, 1]
    [1, 0]


## 3. Kiến thức nền

### 3.1 Điều kiện "nhìn thấy" trên một dãy

Trên dãy `a`, người `i` thấy người `j > i` ⇔ `max(a[i+1..j−1]) < min(a[i], a[j])` (dãy giữa rỗng ⇒ luôn thấy người kề bên).

In [2]:
def sees(a, i, j):
    mid = a[i + 1:j]
    return all(x < min(a[i], a[j]) for x in mid)

a = [3, 1, 4, 2, 5]
print([[j for j in range(i + 1, len(a)) if sees(a, i, j)] for i in range(len(a))])

[[1, 2], [2], [3, 4], [4], []]


### 3.2 Ngăn xếp đơn điệu (monotonic stack)

Duyệt từ phải sang trái, giữ ngăn xếp các chiều cao **tăng dần từ đỉnh xuống đáy** (đỉnh nhỏ nhất). Khi xét người `i`: mọi người trên đỉnh **thấp hơn** `a[i]` đều được `i` nhìn thấy và **bị che** đối với những người bên trái `i` ⇒ pop.

In [3]:
stk = []
for x in [5, 2, 4]:            # duyệt ngược của [4, 2, 5]
    while stk and stk[-1] < x: stk.pop()
    stk.append(x); print("sau khi thêm", x, "->", stk)

sau khi thêm 5 -> [5]
sau khi thêm 2 -> [5, 2]
sau khi thêm 4 -> [5, 4]


### 3.3 Chuyển vị lưới: `zip(*heights)`

Cột `j` của lưới là hàng `j` của ma trận chuyển vị ⇒ xử lý cột bằng chính hàm xử lý hàng.

In [4]:
h = [[5, 1], [3, 1], [4, 1]]
print([list(col) for col in zip(*h)])

[[5, 3, 4], [1, 1, 1]]


### 3.4 Cộng hai ma trận cùng kích thước

In [5]:
A = [[1, 2], [3, 4]]; B = [[10, 20], [30, 40]]
print([[a + b for a, b in zip(ra, rb)] for ra, rb in zip(A, B)])

[[11, 22], [33, 44]]


## 4. Ý tưởng nền tảng / trực giác

**Tách bài:** "nhìn sang phải" chỉ phụ thuộc vào hàng; "nhìn xuống" chỉ phụ thuộc vào cột. Hai tập người được nhìn thấy rời nhau (khác vị trí) ⇒ `answer = rowCount + colCount`.

**Bài 1 chiều:** trên dãy `a`, với mỗi `i`, đếm số `j > i` nhìn thấy được. Đi từ `i` sang phải, giữ `best` = chiều cao lớn nhất đã đi qua (ở giữa):
- người `j` thấy được ⇔ `best < a[j]` (người `j` cao hơn mọi người ở giữa) **và** `best < a[i]` (mọi người giữa thấp hơn `i`);
- khi `best ≥ a[i]` thì dừng (một người cao bằng/hơn `i` đã chắn).

Tính tay `a = [3, 1, 4, 2, 5]`, người `i = 0` (cao 3):

| j | a[j] | best (giữa) | thấy? | ghi chú |
|---|---|---|---|---|
| 1 | 1 | — | ✔ | kề bên |
| 2 | 4 | 1 | ✔ | 1 < 3 và 1 < 4 |
| 3 | 2 | 4 | ✘ | 4 ≥ 3: đã bị chắn, dừng |

⇒ 2. ✔

**Chiều cao bằng nhau:** `a = [2, 2, 2]`: người 0 thấy người 1 (giữa rỗng), không thấy người 2 (người 1 ở giữa không thấp hơn) ⇒ `[1, 1, 0]`.

In [6]:
def count_right_bruteish(a, i):
    best = 0; cnt = 0
    for j in range(i + 1, len(a)):
        if best < a[j] and best < a[i]:
            cnt += 1
        best = max(best, a[j])
        if best >= a[i]: break
    return cnt

for arr in [[3, 1, 4, 2, 5], [2, 2, 2], [5, 1, 3, 1, 4]]:
    print(arr, [count_right_bruteish(arr, i) for i in range(len(arr))])

[3, 1, 4, 2, 5] [2, 1, 2, 1, 0]
[2, 2, 2] [1, 1, 0]
[5, 1, 3, 1, 4] [3, 1, 2, 1, 0]


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn theo đúng định nghĩa: mọi cặp, kiểm tra mọi người ở giữa — O(m·n·(m+n)²)

In [7]:
def brute(heights):
    m, n = len(heights), len(heights[0]); ans = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            h = heights[i][j]
            for c in range(j + 1, n):
                if all(heights[i][k] < min(h, heights[i][c]) for k in range(j + 1, c)): ans[i][j] += 1
            for r in range(i + 1, m):
                if all(heights[k][j] < min(h, heights[r][j]) for k in range(i + 1, r)): ans[i][j] += 1
    return ans

for h, exp in examples:
    got = brute(h)
    print(got, "OK" if got == exp else "SAI")

[[2, 1, 2, 1, 0]] OK
[[3, 1], [2, 1], [1, 0]] OK


### 5.2 Baseline tốt hơn: quét phải với "max ở giữa" — O(m·n·(m+n))

In [8]:
def scan_line(a):
    return [count_right_bruteish(a, i) for i in range(len(a))]

def brute_scan(heights):
    rows = [scan_line(r) for r in heights]
    cols = [scan_line(list(c)) for c in zip(*heights)]
    return [[rows[i][j] + cols[j][i] for j in range(len(heights[0]))] for i in range(len(heights))]

for h, exp in examples:
    print(brute_scan(h), "OK" if brute_scan(h) == exp else "SAI")

[[2, 1, 2, 1, 0]] OK
[[3, 1], [2, 1], [1, 0]] OK


**Độ phức tạp:** mỗi dãy độ dài `L` tốn O(L²) worst (dãy giảm dần: không ai bị chắn sớm) ⇒ tổng `m·n² + n·m²` = `2·400³ ≈ 1,3·10⁸` — chậm trong Python.

**Nút thắt:** khi quét từ mỗi `i`, ta đi qua lại những người mà các `i` bên phải đã "biết". Một người `j` bị che bởi người cao hơn ở bên trái nó thì không ai ở xa hơn bên trái nhìn thấy `j` nữa ⇒ có thể **loại vĩnh viễn** — đó là ngăn xếp đơn điệu.

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt dãy từ **phải sang trái**, ngăn xếp `stk` chứa "những người còn có thể được nhìn thấy từ bên trái", chiều cao **tăng ngặt** từ đỉnh xuống đáy.

Xét người `i` với chiều cao `x`:
1. **Nếu** đỉnh `< x` **thì** `i` nhìn thấy người đó (mọi người giữa họ đã bị pop trước, tức là thấp hơn người đó, và người đó thấp hơn `x`) ⇒ `ans[i] += 1`, pop (người đó bị `i` che vĩnh viễn với bên trái).
2. Sau khi pop hết người thấp hơn: **nếu** ngăn xếp còn người (cao ≥ x) **thì** `i` nhìn thấy đúng **một** người đó (người đầu tiên cao ≥ x) ⇒ `ans[i] += 1`.
3. **Nếu** đỉnh `== x` **thì** pop luôn: người cao bằng `x` bị `i` che (không thấp hơn `i`... chính xác: với người bên trái `k`, người giữa là `i` cao `x`, không thấp hơn người cao `x` kia) ⇒ không ai bên trái thấy họ.
4. Đẩy `x`.

**Bất biến:** sau khi xử lý `i`, ngăn xếp gồm đúng những người `j ≥ i` mà **không** có ai trong `[i, j)` cao ≥ `a[j]` — tức là những người có thể được nhìn thấy từ một vị trí bên trái `i` — xếp với chiều cao tăng ngặt từ đỉnh.

**Khấu hao:** mỗi người được đẩy một lần, pop tối đa một lần ⇒ O(L) cho dãy độ dài `L` ⇒ tổng **O(m·n)**.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Ngăn xếp đơn điệu cho hàng và cột (cách doocs)

In [9]:
def see_line(a):
    n = len(a); res = [0] * n; stk = []
    for i in range(n - 1, -1, -1):
        x = a[i]
        while stk and stk[-1] < x:
            res[i] += 1; stk.pop()
        if stk:
            res[i] += 1                      # người đầu tiên cao >= x
        while stk and stk[-1] == x:
            stk.pop()                        # người cao bằng x bị che
        stk.append(x)
    return res

def mono_stack(heights):
    ans = [see_line(row) for row in heights]
    for j, col in enumerate(zip(*heights)):
        add = see_line(col)
        for i, v in enumerate(add): ans[i][j] += v
    return ans

for h, exp in examples:
    got = mono_stack(h)
    print(got, "OK" if got == exp else "SAI")
print(see_line([2, 2, 2]), see_line([3, 1, 4, 2, 5]), see_line([1, 2, 3]), see_line([3, 2, 1]))

[[2, 1, 2, 1, 0]] OK
[[3, 1], [2, 1], [1, 0]] OK
[1, 1, 0] [2, 1, 2, 1, 0] [1, 1, 0] [1, 1, 0]


**Độ phức tạp cách 2:** O(m·n) thời gian, O(m·n) cho đáp án + O(max(m, n)) cho ngăn xếp.

### 7.2 Cách 3 — Ngăn xếp duyệt **trái sang phải** ("ai nhìn thấy tôi?")

Đảo vai: khi xét người `j`, những người bên trái nhìn thấy `j` là: các người bị `j` pop (thấp hơn `j`, họ nhìn thấy `j` vì giữa họ và `j` toàn người thấp hơn họ), cộng người đầu tiên còn lại cao hơn/bằng... — ta cộng `+1` vào **họ**. Cách này cho cùng kết quả, minh họa ngăn xếp có thể chạy theo cả hai chiều.

In [10]:
def see_line_left_to_right(a):
    n = len(a); res = [0] * n; stk = []            # stk chứa chỉ số, chiều cao giảm ngặt từ đáy lên?
    for j in range(n):
        x = a[j]
        while stk and a[stk[-1]] < x:              # những người thấp hơn x nhìn thấy x rồi bị che
            res[stk.pop()] += 1
        if stk:
            res[stk[-1]] += 1                      # người gần nhất cao >= x nhìn thấy x
            if a[stk[-1]] == x:
                stk.pop()                          # người cao bằng x không nhìn xuyên qua x được nữa
        stk.append(j)
    return res

for arr in [[3, 1, 4, 2, 5], [2, 2, 2], [5, 1, 3, 1, 4], [4, 4, 1, 4]]:
    print(arr, see_line_left_to_right(arr), see_line(arr))

[3, 1, 4, 2, 5] [2, 1, 2, 1, 0] [2, 1, 2, 1, 0]
[2, 2, 2] [1, 1, 0] [1, 1, 0]
[5, 1, 3, 1, 4] [3, 1, 2, 1, 0] [3, 1, 2, 1, 0]
[4, 4, 1, 4] [1, 2, 1, 0] [1, 2, 1, 0]


Ở chiều này, ngăn xếp lưu **chỉ số** để cộng vào đúng người. Mỗi người được đẩy và pop tối đa một lần ⇒ O(L).

### 7.3 Cách 4 — Dùng chung một hàm và chuyển vị

In [11]:
def via_transpose(heights):
    rows = [see_line(r) for r in heights]
    cols_t = [see_line(list(c)) for c in zip(*heights)]       # kết quả của cột, dạng chuyển vị
    cols = [list(r) for r in zip(*cols_t)]                    # chuyển vị ngược về
    return [[a + b for a, b in zip(ra, rb)] for ra, rb in zip(rows, cols)]

for h, exp in examples:
    print(via_transpose(h), "OK" if via_transpose(h) == exp else "SAI")

[[2, 1, 2, 1, 0]] OK
[[3, 1], [2, 1], [1, 0]] OK


## 8. TRACE từng bước

### 8.1 Trace ngăn xếp trên dãy `[3, 1, 4, 2, 5]` (duyệt phải sang trái)

| i | x | pop (thấp hơn x) | còn người cao ≥ x? | pop (bằng x) | res[i] | stk sau |
|---|---|---|---|---|---|---|
| 4 | 5 | — | không | — | 0 | [5] |
| 3 | 2 | — | có (5) | — | 1 | [5, 2] |
| 2 | 4 | 2 | có (5) | — | 2 | [5, 4] |
| 1 | 1 | — | có (4) | — | 1 | [5, 4, 1] |
| 0 | 3 | 1 | có (4) | — | 2 | [5, 4, 3] |

In [12]:
def trace_line(a):
    stk = []; res = [0] * len(a)
    for i in range(len(a) - 1, -1, -1):
        x = a[i]; popped = []
        while stk and stk[-1] < x:
            popped.append(stk.pop()); res[i] += 1
        blocker = stk[-1] if stk else None
        if stk: res[i] += 1
        eq = []
        while stk and stk[-1] == x: eq.append(stk.pop())
        stk.append(x)
        print(f"i={i} x={x}: thấy (thấp hơn) {popped}, thấy người chắn {blocker}, bỏ bằng nhau {eq} -> res={res[i]}, stk={stk}")
    return res

print("kết quả:", trace_line([3, 1, 4, 2, 5]))

i=4 x=5: thấy (thấp hơn) [], thấy người chắn None, bỏ bằng nhau [] -> res=0, stk=[5]
i=3 x=2: thấy (thấp hơn) [], thấy người chắn 5, bỏ bằng nhau [] -> res=1, stk=[5, 2]
i=2 x=4: thấy (thấp hơn) [2], thấy người chắn 5, bỏ bằng nhau [] -> res=2, stk=[5, 4]
i=1 x=1: thấy (thấp hơn) [], thấy người chắn 4, bỏ bằng nhau [] -> res=1, stk=[5, 4, 1]
i=0 x=3: thấy (thấp hơn) [1], thấy người chắn 4, bỏ bằng nhau [] -> res=2, stk=[5, 4, 3]
kết quả: [2, 1, 2, 1, 0]


### 8.2 Trace với chiều cao bằng nhau `[4, 4, 1, 4]`

In [13]:
print("kết quả:", trace_line([4, 4, 1, 4]))
print("brute  :", [sum(1 for j in range(i + 1, 4) if sees([4, 4, 1, 4], i, j)) for i in range(4)])

i=3 x=4: thấy (thấp hơn) [], thấy người chắn None, bỏ bằng nhau [] -> res=0, stk=[4]
i=2 x=1: thấy (thấp hơn) [], thấy người chắn 4, bỏ bằng nhau [] -> res=1, stk=[4, 1]
i=1 x=4: thấy (thấp hơn) [1], thấy người chắn 4, bỏ bằng nhau [4] -> res=2, stk=[4]
i=0 x=4: thấy (thấp hơn) [], thấy người chắn 4, bỏ bằng nhau [4] -> res=1, stk=[4]
kết quả: [1, 2, 1, 0]
brute  : [1, 2, 1, 0]


Người 0 (cao 4) thấy người 1 (kề bên, cao 4) nhưng không thấy người 3 — người 1 ở giữa không thấp hơn. Người 1 thấy người 2 (cao 1) và người 3 (cao 4, người giữa cao 1 thấp hơn cả hai).

### 8.3 Ghép hàng và cột cho ví dụ 2

In [14]:
h = examples[1][0]
print("hàng:", [see_line(r) for r in h])
print("cột :", [see_line(list(c)) for c in zip(*h)], "(mỗi dòng là một cột)")
print("tổng:", mono_stack(h))

hàng: [[1, 0], [1, 0], [1, 0]]
cột : [[2, 1, 0], [1, 1, 0]] (mỗi dòng là một cột)
tổng: [[3, 1], [2, 1], [1, 0]]


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def seePeople(self, heights: List[List[int]]) -> List[List[int]]:
        def see(a: List[int]) -> List[int]:
            n = len(a)
            res = [0] * n
            stk = []                                  # chiều cao tăng ngặt từ đỉnh xuống đáy
            for i in range(n - 1, -1, -1):
                x = a[i]
                while stk and stk[-1] < x:            # thấy và che những người thấp hơn
                    res[i] += 1
                    stk.pop()
                if stk:                               # thấy người đầu tiên cao >= x
                    res[i] += 1
                while stk and stk[-1] == x:           # người cao bằng x bị che với bên trái
                    stk.pop()
                stk.append(x)
            return res

        m, n = len(heights), len(heights[0])
        ans = [see(row) for row in heights]
        for j in range(n):
            col = see([heights[i][j] for i in range(m)])
            for i in range(m):
                ans[i][j] += col[i]
        return ans

In [16]:
sol = Solution()
for h, exp in examples:
    got = sol.seePeople(h)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

[[2, 1, 2, 1, 0]] | kỳ vọng [[2, 1, 2, 1, 0]] | OK
[[3, 1], [2, 1], [1, 0]] | kỳ vọng [[3, 1], [2, 1], [1, 0]] | OK


## 10. Vì sao thuật toán đúng

Xét một dãy `a`. Ta chứng minh `see(a)[i]` = số `j > i` mà `i` nhìn thấy.

**Bất biến (I):** ngay trước khi xử lý `i`, ngăn xếp chứa đúng các chỉ số `j > i` sao cho **không** có `k ∈ (i, j)` với `a[k] ≥ a[j]` (gọi là "lộ diện từ `i`"), xếp theo `j` tăng từ đỉnh xuống đáy, và chiều cao tăng **ngặt** từ đỉnh xuống đáy.

**Người `i` thấy `j` ⇔ `j` lộ diện từ `i` và mọi người giữa thấp hơn `a[i]`.** Trong các người lộ diện (chiều cao tăng dần theo khoảng cách), những người thấp hơn `a[i]` đều thỏa (người giữa họ và `i` đều thấp hơn họ, nên thấp hơn `a[i]`); người lộ diện đầu tiên có chiều cao ≥ `a[i]` cũng thỏa (người giữa đều thấp hơn nó và đều thấp hơn `a[i]` vì họ hoặc không lộ diện — bị che bởi một người lộ diện thấp hơn `a[i]` — hoặc là người lộ diện thấp hơn `a[i]`); mọi người lộ diện xa hơn bị người đó chắn (nó không thấp hơn `a[i]`). Không người không-lộ-diện nào được thấy (có người ở giữa cao ≥ họ). ⇒ số người `i` thấy = (số phần tử `< a[i]` trên đỉnh ngăn xếp) + [còn phần tử]. Đó chính là thao tác của thuật toán.

**Duy trì (I):** sau khi thêm `i`, những người thấp hơn hoặc **bằng** `a[i]` không còn lộ diện đối với bất kỳ vị trí nào bên trái `i` (có `i` ở giữa, cao ≥ họ) ⇒ pop đúng; `i` luôn lộ diện ⇒ push. Chiều cao vẫn tăng ngặt. ∎

**Ghép hàng/cột:** tập người nhìn thấy theo hàng (bên phải) và theo cột (bên dưới) rời nhau ⇒ cộng.

## 11. Phân tích Time / Space Complexity

`m, n ≤ 400`.

| Cách | Time | Space |
|---|---|---|
| 1a. Mọi cặp + kiểm tra giữa | O(mn(m+n)²) | O(mn) |
| 1b. Quét với max ở giữa | O(mn(m+n)) | O(mn) |
| 2. Ngăn xếp phải → trái | O(mn) | O(mn) |
| 3. Ngăn xếp trái → phải | O(mn) | O(mn) |
| 4. Chuyển vị | O(mn) | O(mn) |

In [17]:
rng = random.Random(2282)
for name_d, grid in [("ngẫu nhiên 400x400", [[rng.randint(1, 10**5) for _ in range(400)] for _ in range(400)]),
                     ("giảm dần (worst cho quét)", [[400 * 400 - (i * 400 + j) for j in range(400)] for i in range(400)])]:
    res = []
    for name, f in [("mono_stack", mono_stack), ("via_transpose", via_transpose), ("Solution", Solution().seePeople)]:
        t0 = time.perf_counter(); res.append(f(grid)); dt = time.perf_counter() - t0
        print(f"{name_d:<26} {name:<13} {dt*1000:6.0f} ms")
    assert all(r == res[0] for r in res)
small = [[rng.randint(1, 10**5) for _ in range(120)] for _ in range(120)]
t0 = time.perf_counter(); brute_scan(small); print(f"brute_scan 120x120: {(time.perf_counter()-t0)*1000:.0f} ms (400x400 sẽ chậm hơn ~37 lần)")

ngẫu nhiên 400x400         mono_stack        70 ms


ngẫu nhiên 400x400         via_transpose     56 ms
ngẫu nhiên 400x400         Solution          58 ms
giảm dần (worst cho quét)  mono_stack        43 ms
giảm dần (worst cho quét)  via_transpose     46 ms


giảm dần (worst cho quét)  Solution          44 ms
brute_scan 120x120: 12 ms (400x400 sẽ chậm hơn ~37 lần)


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng — đối chiếu lại đề gốc nếu có Premium)

In [18]:
ALL = {"brute": brute, "brute_scan": brute_scan, "mono_stack": mono_stack, "via_transpose": via_transpose,
       "lr_stack": lambda h: [[a + b for a, b in zip(ra, rb)] for ra, rb in
                             zip([see_line_left_to_right(r) for r in h],
                                 [list(r) for r in zip(*[see_line_left_to_right(list(c)) for c in zip(*h)])])],
       "Solution": Solution().seePeople}
edge = examples + [
    ([[7]], [[0]]),
    ([[1, 2, 3]], [[1, 1, 0]]),                  # tăng dần: chỉ thấy người kề
    ([[3, 2, 1]], [[1, 1, 0]]),                  # giảm dần: 2 ở giữa không thấp hơn 1 nên 3 không thấy 1
    ([[2, 2, 2]], [[1, 1, 0]]),
    ([[4, 4, 1, 4]], [[1, 2, 1, 0]]),
    ([[1], [2], [1]], [[1], [1], [0]]),
]
for h, exp in edge:
    ref = brute(h)
    for name, f in ALL.items():
        assert f([r[:] for r in h]) == ref, (name, h, f(h), ref)
    print(h, "->", ref, "| ghi trong bảng:", exp, "| khớp:" , ref == exp)

[[3, 1, 4, 2, 5]] -> [[2, 1, 2, 1, 0]] | ghi trong bảng: [[2, 1, 2, 1, 0]] | khớp: True
[[5, 1], [3, 1], [4, 1]] -> [[3, 1], [2, 1], [1, 0]] | ghi trong bảng: [[3, 1], [2, 1], [1, 0]] | khớp: True
[[7]] -> [[0]] | ghi trong bảng: [[0]] | khớp: True
[[1, 2, 3]] -> [[1, 1, 0]] | ghi trong bảng: [[1, 1, 0]] | khớp: True
[[3, 2, 1]] -> [[1, 1, 0]] | ghi trong bảng: [[1, 1, 0]] | khớp: True
[[2, 2, 2]] -> [[1, 1, 0]] | ghi trong bảng: [[1, 1, 0]] | khớp: True
[[4, 4, 1, 4]] -> [[1, 2, 1, 0]] | ghi trong bảng: [[1, 2, 1, 0]] | khớp: True
[[1], [2], [1]] -> [[1], [1], [0]] | ghi trong bảng: [[1], [1], [0]] | khớp: True


Chú ý case `[[3, 2, 1]]` (giảm dần): người cao 3 thấy người cao 2 (kề bên) **và** người cao 1? Người giữa (cao 2) phải thấp hơn **cả hai** (3 và 1) — 2 không thấp hơn 1 ⇒ **không** thấy. Đáp án `[[1, 1, 0]]`. Đây là chỗ hay nhầm: điều kiện là "thấp hơn **cả hai**", không chỉ thấp hơn người nhìn. (Cell trên in `khớp: True/False` để phát hiện kỳ vọng ghi tay sai.)

### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

Chiều cao lấy từ tập nhỏ để có nhiều trường hợp bằng nhau.

In [19]:
rng = random.Random(42)
for _ in range(500):
    m, n = rng.randint(1, 6), rng.randint(1, 6); H = rng.choice([2, 3, 5, 10**5])
    h = [[rng.randint(1, H) for _ in range(n)] for _ in range(m)]
    ref = brute(h)
    for name, f in ALL.items():
        assert f([r[:] for r in h]) == ref, (name, h)
print("500 lưới ngẫu nhiên OK")

500 lưới ngẫu nhiên OK


### 12.3 Tính chất trên một dãy: tổng số cặp nhìn thấy ≤ 2(L − 1); mỗi người thấy ít nhất người kề bên (nếu có)

In [20]:
rng = random.Random(7)
for _ in range(500):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 30))]
    r = see_line(a)
    assert all(r[i] >= 1 for i in range(len(a) - 1)) and r[-1] == 0
    assert sum(r) <= 2 * (len(a) - 1)
print("tính chất OK")

tính chất OK


Vì sao `≤ 2(L − 1)`? Mỗi lần pop đóng góp 1 (≤ L − 1 lần pop vì mỗi người pop tối đa một lần), cộng tối đa 1 "người chắn" cho mỗi `i` (≤ L − 1) ⇒ tổng ≤ 2(L − 1) — đây cũng là bằng chứng cho khấu hao O(L).

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không xử lý chiều cao bằng nhau (không pop các phần tử bằng `x`)

In [21]:
def wrong_no_eq_pop(a):
    n = len(a); res = [0] * n; stk = []
    for i in range(n - 1, -1, -1):
        x = a[i]
        while stk and stk[-1] < x: res[i] += 1; stk.pop()
        if stk: res[i] += 1
        stk.append(x)                                  # SAI: người cao bằng x vẫn nằm lại
    return res

print("bản sai :", wrong_no_eq_pop([2, 2, 2]), "| bản đúng:", see_line([2, 2, 2]), "(trùng hợp: dãy quá ngắn nên chưa lộ lỗi)")
print("bản sai :", wrong_no_eq_pop([5, 2, 2, 3]), "| bản đúng:", see_line([5, 2, 2, 3]))

bản sai : [1, 1, 0] | bản đúng: [1, 1, 0] (trùng hợp: dãy quá ngắn nên chưa lộ lỗi)
bản sai : [3, 1, 1, 0] | bản đúng: [2, 1, 1, 0]


**Vì sao sai:** ngăn xếp không còn tăng **ngặt**: với `[5, 2, 2, 3]`, khi xét người cao 5, ngăn xếp có `[3, 2, 2]` (đỉnh là 2) — người 5 pop cả hai số 2 và đếm 2 lần, trong khi người cao 2 thứ hai bị người cao 2 thứ nhất che. **Sửa:** sau khi đếm, pop các phần tử bằng `x`.

### Lỗi 2 — Dùng `<=` khi pop (gộp người bằng chiều cao vào nhóm "thấp hơn")

In [22]:
def wrong_le_pop(a):
    n = len(a); res = [0] * n; stk = []
    for i in range(n - 1, -1, -1):
        x = a[i]
        while stk and stk[-1] <= x: res[i] += 1; stk.pop()   # pop cả bằng, đếm như "thấp hơn"
        if stk: res[i] += 1                                # SAI: đếm thêm người chắn sau người bằng x
        stk.append(x)
    return res

print("bản sai :", wrong_le_pop([2, 1, 2, 3]), "| bản đúng:", see_line([2, 1, 2, 3]))

bản sai : [3, 1, 1, 0] | bản đúng: [2, 1, 1, 0]


**Vì sao sai:** người cao bằng `x` **chắn** tầm nhìn của `x` (người ở giữa phải thấp hơn cả hai — người cao bằng `x` không thấp hơn `x`). Đếm người bằng `x` rồi lại đếm thêm "người chắn" phía sau là đếm thừa: `[2, 1, 2, 3]` — người đầu thấy 1 và người cao 2, **không** thấy người cao 3. **Sửa:** pop `<` để đếm, kiểm tra `if stk` cho người chắn (có thể là người bằng `x`), rồi mới pop `==`.

### Lỗi 3 — Chỉ kiểm tra người ở giữa thấp hơn **người nhìn** (quên "cả hai")

In [23]:
def wrong_one_side(a):
    res = []
    for i in range(len(a)):
        cnt = 0; best = 0
        for j in range(i + 1, len(a)):
            if best < a[i]: cnt += 1                    # SAI: không cần best < a[j]
            best = max(best, a[j])
            if best >= a[i]: break
        res.append(cnt)
    return res

print("bản sai :", wrong_one_side([3, 2, 1]), "| bản đúng:", see_line([3, 2, 1]))

bản sai : [2, 1, 0] | bản đúng: [1, 1, 0]


**Vì sao sai:** người cao 3 không thấy người cao 1 vì người cao 2 ở giữa **không thấp hơn 1**. **Sửa:** điều kiện `max(giữa) < min(a[i], a[j])`.

### Lỗi 4 — Cộng cả hướng trái / trên (nhìn 4 hướng)

In [24]:
def wrong_four_dirs(heights):
    rows = [see_line(r) for r in heights]; rows_rev = [see_line(r[::-1])[::-1] for r in heights]
    cols = [list(r) for r in zip(*[see_line(list(c)) for c in zip(*heights)])]
    return [[rows[i][j] + rows_rev[i][j] + cols[i][j] for j in range(len(heights[0]))] for i in range(len(heights))]

print("bản sai :", wrong_four_dirs([[3, 1, 4, 2, 5]]), "| bản đúng:", Solution().seePeople([[3, 1, 4, 2, 5]]))

bản sai : [[2, 2, 4, 2, 2]] | bản đúng: [[2, 1, 2, 1, 0]]


**Sửa:** chỉ **phải** và **dưới** theo đề.

### Lỗi 5 — Ghép cột sai chỉ số (cộng `col[j]` thay vì `col[i]`)

In [25]:
def wrong_index(heights):
    m, n = len(heights), len(heights[0]); ans = [see_line(r) for r in heights]
    for j in range(n):
        col = see_line([heights[i][j] for i in range(m)])
        for i in range(m):
            ans[i][j] += col[j] if j < m else 0           # SAI: lẫn chỉ số hàng/cột
    return ans

h = [[5, 1, 2], [3, 1, 2], [4, 1, 2]]
print("bản sai :", wrong_index(h), "| bản đúng:", Solution().seePeople(h))

bản sai : [[4, 2, 0], [4, 2, 0], [4, 2, 0]] | bản đúng: [[4, 2, 1], [3, 2, 1], [2, 1, 0]]


**Sửa:** kết quả cột `j` là danh sách theo **hàng** `i` ⇒ `ans[i][j] += col[i]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nhìn thấy / đứng trước người cao hơn / phần tử lớn hơn kế tiếp" ⇒ **ngăn xếp đơn điệu**.
- Bài 2D chỉ nhìn theo hàng và cột ⇒ **tách thành các dãy 1D**, dùng chung một hàm (chuyển vị bằng `zip(*grid)`).
- Cẩn thận "nhỏ hơn **cả hai**" và chiều cao **bằng nhau** — viết vét cạn để đối chiếu.
- Khấu hao: mỗi phần tử vào/ra ngăn xếp một lần ⇒ O(n).
- Bài cùng họ: 1944 Number of Visible People in a Queue (bản 1D, phần tử phân biệt), 496/503 Next Greater Element, 84 Largest Rectangle in Histogram, 739 Daily Temperatures.

## 15. Biến thể của bài

### 15.1 Biến thể A — LC 1944 (1 dãy, chiều cao **phân biệt**)

Khi không có bằng nhau, vòng pop `==` là thừa; hàm `see_line` vẫn đúng.

In [26]:
def can_see_persons_count(heights):
    return see_line(heights)
print(can_see_persons_count([10, 6, 8, 5, 11, 9]), "(LC 1944 ví dụ: [3,1,2,1,1,0])")

[3, 1, 2, 1, 1, 0] (LC 1944 ví dụ: [3,1,2,1,1,0])


### 15.2 Biến thể B — Nhìn được cả 4 hướng

In [27]:
def see_four_dirs(heights):
    m, n = len(heights), len(heights[0])
    R = [see_line(r) for r in heights]
    L = [see_line(r[::-1])[::-1] for r in heights]
    D = [list(r) for r in zip(*[see_line(list(c)) for c in zip(*heights)])]
    U = [list(r) for r in zip(*[see_line(list(c)[::-1])[::-1] for c in zip(*heights)])]
    return [[R[i][j] + L[i][j] + D[i][j] + U[i][j] for j in range(n)] for i in range(m)]

def see_four_brute(heights):
    m, n = len(heights), len(heights[0]); out = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            row = heights[i]; col = [heights[k][j] for k in range(m)]
            out[i][j] = sum(sees(row, min(j, c), max(j, c)) for c in range(n) if c != j) + \
                        sum(sees(col, min(i, r), max(i, r)) for r in range(m) if r != i)
    return out

rng = random.Random(152)
for _ in range(200):
    h = [[rng.randint(1, 4) for _ in range(rng.randint(1, 5))]]
    h += [[rng.randint(1, 4) for _ in range(len(h[0]))] for _ in range(rng.randint(0, 4))]
    assert see_four_dirs(h) == see_four_brute(h)
print(see_four_dirs([[3, 1, 4, 2, 5]]), "- khớp brute force")

[[2, 2, 4, 2, 2]] - khớp brute force


Hàm `sees(a, i, j)` đối xứng theo nghĩa "i thấy j ⇔ j thấy i" (điều kiện chỉ phụ thuộc vào hai đầu và phần giữa) ⇒ nhìn sang trái = đảo dãy rồi nhìn sang phải.

### 15.3 Biến thể C — Tổng số **cặp** nhìn thấy nhau trên mỗi hàng (đếm cặp không có thứ tự)

Bằng tổng `see_line` (mỗi cặp được đếm đúng một lần từ người bên trái).

In [28]:
def pairs_in_row(a):
    return sum(see_line(a))
def pairs_brute(a):
    return sum(1 for i in range(len(a)) for j in range(i + 1, len(a)) if sees(a, i, j))
rng = random.Random(153)
for _ in range(300):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 15))]
    assert pairs_in_row(a) == pairs_brute(a)
print(pairs_in_row([3, 1, 4, 2, 5]), "cặp")

6 cặp


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Mọi cặp | kiểm tra người giữa | O(mn(m+n)²) | O(mn) | kiểm chứng |
| 1b. Quét max | dừng khi bị chắn | O(mn(m+n)) | O(mn) | baseline |
| 2. Ngăn xếp phải→trái | đếm người lộ diện | O(mn) | O(mn) | **mặc định** |
| 3. Ngăn xếp trái→phải | "ai thấy tôi" | O(mn) | O(mn) | luyện tư duy đảo vai |
| 4. Chuyển vị | một hàm cho cả hàng và cột | O(mn) | O(mn) | code gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `seePeople([[1, 1, 1, 1]])` và `seePeople([[2, 1, 1, 2]])`?

**Bài 2 (tính tay):** Trace ngăn xếp cho dãy `[5, 3, 4, 3, 6]`.

**Bài 3 (điền TODO):** Viết `bt3_todo(a)` trả về với mỗi `i` **danh sách chỉ số** những người `i` nhìn thấy (bên phải), dùng ngăn xếp lưu chỉ số.

**Bài 4 (tự viết lại):** Viết `bt4_todo(heights)` cài bản nộp không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(a)` trả về với mỗi người số người **bên trái** nhìn thấy họ (tức "bao nhiêu người thấy tôi từ bên trái") — một lượt ngăn xếp trái → phải.

In [29]:
def bt3_todo(a):
    # TODO: danh sách chỉ số người i nhìn thấy
    pass

def bt4_todo(heights):
    # TODO: bản nộp
    pass

def bt5_todo(a):
    # TODO: số người bên trái nhìn thấy mỗi người
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: bằng nhau ⇒ chỉ thấy người kề. Mức 2: `[1, 1, 1, 0]`. Mức 3: `[2,1,1,2]`: người 0 thấy 1 (kề) và người 3? giữa là `1, 1` thấp hơn 2 và 2 ⇒ thấy; người 0 thấy người 2? giữa là người 1 cao 1, không thấp hơn người 2 (cao 1) ⇒ không ⇒ `[2, 1, 1, 0]`.

**Bài 2.** Mức 1: từ phải: 6 vào. Mức 2: 3: thấy 6 → 1; 4: pop 3, thấy 6 → 2; 3: thấy 4 → 1. Mức 3: 5: pop 3, pop 4, thấy 6 → 3 ⇒ `[3, 1, 2, 1, 0]`.

**Bài 3.** Mức 1: ngăn xếp lưu chỉ số. Mức 2: mỗi lần pop ghi chỉ số đó vào danh sách của `i`. Mức 3: thêm chỉ số đỉnh còn lại (nếu có).

**Bài 4.** Mức 1: hàm `see`. Mức 2: áp cho hàng. Mức 3: áp cho cột, cộng `col[i]` vào `ans[i][j]`.

**Bài 5.** Mức 1: dùng cách 3 nhưng đếm cho người **được** thấy. Mức 2: khi xét `j`, những người bị pop (thấp hơn) đều thấy `j`; người còn lại trên đỉnh cũng thấy `j`. Mức 3: `res[j] = số pop + [stk khác rỗng]`; pop thêm người bằng `a[j]`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `[[1, 1, 1, 0]]`, `[[2, 1, 1, 0]]`. **Bài 2:** `[3, 1, 2, 1, 0]`.

In [30]:
assert Solution().seePeople([[1, 1, 1, 1]]) == [[1, 1, 1, 0]]
assert Solution().seePeople([[2, 1, 1, 2]]) == [[2, 1, 1, 0]]
assert see_line([5, 3, 4, 3, 6]) == [3, 1, 2, 1, 0]

def bt3(a):
    n = len(a); out = [[] for _ in range(n)]; stk = []
    for i in range(n - 1, -1, -1):
        while stk and a[stk[-1]] < a[i]: out[i].append(stk.pop())
        if stk: out[i].append(stk[-1])
        while stk and a[stk[-1]] == a[i]: stk.pop()
        stk.append(i)
    return [sorted(x) for x in out]

def bt4(heights):
    return Solution().seePeople(heights)

def bt5(a):
    n = len(a); res = [0] * n; stk = []
    for j in range(n):
        while stk and a[stk[-1]] < a[j]: stk.pop(); res[j] += 1
        if stk:
            res[j] += 1
            while stk and a[stk[-1]] == a[j]: stk.pop()
        stk.append(j)
    return res

rng = random.Random(1982)
for _ in range(500):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 15))]
    assert bt3(a) == [[j for j in range(i + 1, len(a)) if sees(a, i, j)] for i in range(len(a))]
    assert bt5(a) == [sum(1 for i in range(j) if sees(a, i, j)) for j in range(len(a))]
    h = [a, [rng.randint(1, 5) for _ in a]]
    assert bt4(h) == brute(h)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def see(a):
    res, stk = [0] * len(a), []
    for i in range(len(a) - 1, -1, -1):
        while stk and stk[-1] < a[i]: res[i] += 1; stk.pop()   # thấy + che người thấp hơn
        if stk: res[i] += 1                                    # thấy người chắn đầu tiên
        while stk and stk[-1] == a[i]: stk.pop()               # người bằng mình bị che
        stk.append(a[i])
    return res
# answer = see(hàng) + see(cột)
```

**Checklist:**
- [ ] Chỉ nhìn **phải** và **dưới**.
- [ ] Người giữa thấp hơn **cả hai**.
- [ ] Pop `<` để đếm; đếm người chắn nếu còn; pop `==`; push.
- [ ] Ghép cột đúng chỉ số (`ans[i][j] += col[i]`).
- [ ] (Premium) Đối chiếu lại đề gốc.